In [22]:
from ugot import ugot

got = ugot.UGOT()
got.initialize("192.168.88.1")
got.show_light_rgb([1, 2, 3], 255, 255, 255)
got.play_audio_tts("Connection successful")

192.168.88.1:50051


In [24]:
got.load_models(['color_recognition'])

True

In [ ]:
# XYZ Movement
got.mecanum_translate_speed(45, 20) # Moves 45 degrees, 20 cm/s speed
got.mecanum_translate_speed(-90, 20) # Move -90 (left) degrees, 20 cm/s speed

In [ ]:
# Mechanical arm
got.mechanical_joint_control(10, 0, 0, 1000) # joint 1 10 degrees, joint 2 0 degrees, joint 3 0 degrees

## Centralize Cube

In [26]:
# How to centralize with the cube
import cv2
import numpy as np

got.open_camera()
try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()

        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Get color recognition result: [color, shape, center_x, center_y, height, width, area]
        result = got.get_color_total_info() 

        if result:
            # Unpack into several variables
            color, shape, cx, cy, h, w, area = result
            cx, cy = int(cx), int(cy)
            half_w, half_h = int(w / 2), int(h / 2)

            # Compute bounding box corners
            x1, y1 = cx - half_w, cy - half_h
            x2, y2 = cx + half_w, cy + half_h

            # -------------------------------------------------
            # Centralize logic
            # -------------------------------------------------
            if cx < 270:
                got.mecanum_translate_speed(-90, 20)
                action = "Moving Left"
            elif cx > 370:
                got.mecanum_translate_speed(90, 20)
                action = "Moving Right"
            else:
                got.mecanum_stop()
                action = "Stopped"

            # Draw bounding box and center dot
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.circle(img, (cx, cy), 4, (0, 0, 255), -1)

            # Label with color, shape, and area
            lines = [
                f"{color} {shape} width: {w}, area: {area}",
                f"cx: {cx} cy: {cy}",
                f"Action: {action}"
            ]

            line_height = 25  # adjust based on font scale
            for i, line in enumerate(lines):
                y = (y1 - 10) - (len(lines) - 1 - i) * line_height
                cv2.putText(img, line, (x1, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        # Display the frame
        cv2.imshow("Colour Recognition", img)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Closing camera window...")
            break

finally:
    got.mecanum_stop()

    cv2.destroyAllWindows()
    cv2.waitKey(1)

Closing camera window...


## Approach and Pick up Cube

In [28]:
# How to approach the cube
import cv2
import time
import numpy as np

pick_up_cube = False
got.open_camera()
try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()

        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Get color recognition result: [color, shape, center_x, center_y, height, width, area]
        result = got.get_color_total_info() 

        if result:
            # Unpack into several variables
            color, shape, cx, cy, h, w, area = result
            cx, cy = int(cx), int(cy)
            half_w, half_h = int(w / 2), int(h / 2)

            # Compute bounding box corners
            x1, y1 = cx - half_w, cy - half_h
            x2, y2 = cx + half_w, cy + half_h

            # -------------------------------------------------
            # Approach logic
            # -------------------------------------------------
            if w < 180:
                got.mecanum_translate_speed(0, 20)
                action = "Approaching"
            else:
                got.mecanum_stop()
                action = "Stopped"
                pick_up_cube = True

            if pick_up_cube == True:
                got.mechanical_clamp_release()
                time.sleep(1)
                got.mechanical_joint_control(int((320 - cx) * 0.1), -50, -20, 1000)
                time.sleep(1)
                got.mechanical_clamp_close()
                time.sleep(1)
                got.mechanical_joint_control(0, 90, 90, 1000)
                pick_up_cube = False

            # Draw bounding box and center dot
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.circle(img, (cx, cy), 4, (0, 0, 255), -1)

            # Label with color, shape, and area
            lines = [
                f"{color} {shape} width: {w}, area: {area}",
                f"cx: {cx} cy: {cy}",
                f"Action: {action}"
            ]

            line_height = 25  # adjust based on font scale
            for i, line in enumerate(lines):
                y = (y1 - 10) - (len(lines) - 1 - i) * line_height
                cv2.putText(img, line, (x1, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        # Display the frame
        cv2.imshow("Colour Recognition", img)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Closing camera window...")
            break

finally:
    got.mecanum_stop()

    cv2.destroyAllWindows()
    cv2.waitKey(1)

Closing camera window...
